# 🚀 Fake News Detection - Model Training (Google Colab)

This notebook allows you to train the DistilBERT model using a free GPU on Google Colab.

## 1. Setup Environment

In [ ]:
# Force install specific compatible versions
# datasets==2.16.0 is the last version that reliably supports legacy scripts like LIAR
!pip install "datasets==2.16.0" "transformers[torch]" "accelerate>=0.21.0" scikit-learn pandas mlflow lime shap

## 2. Create Project Structure
We will create the necessary directories and write the training scripts directly into the Colab environment.

In [ ]:
import os
os.makedirs('src', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('models', exist_ok=True)

In [ ]:
%%writefile src/preprocess.py
import pandas as pd
import re
import string
from datasets import load_dataset
import os
from sklearn.model_selection import train_test_split

def clean_text(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'<[^>]*>', '', text)
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess_data():
    print("Loading reliable fake news dataset from Hugging Face...")
    try:
        dataset = load_dataset("GonzaloA/fake_news", trust_remote_code=True)
        df = pd.DataFrame(dataset['train'])
    except Exception:
        dataset = load_dataset("mosharaf2k/fake-news-dataset", trust_remote_code=True)
        df = pd.DataFrame(dataset['train'])
        if 'text' not in df.columns: df = df.rename(columns={'statement': 'text'})

    df = df.dropna(subset=['text', 'label'])
    df['statement'] = df['text'].apply(clean_text)
    df['label'] = df['label'].astype(int)

    train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df['label'])
    val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['label'])

    os.makedirs('data/processed', exist_ok=True)
    cols = ['statement', 'label']
    train_df[cols].to_csv('data/processed/train.csv', index=False)
    val_df[cols].to_csv('data/processed/val.csv', index=False)
    test_df[cols].to_csv('data/processed/test.csv', index=False)
    print("Preprocessing complete!")

if __name__ == "__main__":
    preprocess_data()

In [ ]:
%%writefile src/train.py
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

MODEL_NAME = "distilbert-base-uncased"
BATCH_SIZE = 16
MAX_LEN = 128
EPOCHS = 3
LR = 2e-5

class NewsDataset(Dataset):
    def __init__(self, statements, labels, tokenizer, max_len):
        self.statements, self.labels, self.tokenizer, self.max_len = statements, labels, tokenizer, max_len
    def __len__(self): return len(self.statements)
    def __getitem__(self, item):
        encoding = self.tokenizer(
            str(self.statements[item]),
            add_special_tokens=True, max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt',
        )
        return {'input_ids': encoding['input_ids'].flatten(), 'attention_mask': encoding['attention_mask'].flatten(), 'labels': torch.tensor(self.labels[item], dtype=torch.long)}

def eval_model(model, data_loader, device):
    model.eval(); all_preds, all_labels = [], []
    with torch.no_grad():
        for d in data_loader:
            input_ids, mask = d["input_ids"].to(device), d["attention_mask"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=mask)
            all_preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy()); all_labels.extend(d["labels"].numpy())
    return precision_recall_fscore_support(all_labels, all_preds, average='weighted')[2]

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on {device}")
    
    train_df, val_df = pd.read_csv('data/processed/train.csv'), pd.read_csv('data/processed/val.csv')
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    train_loader = DataLoader(NewsDataset(train_df.statement.to_numpy(), train_df.label.to_numpy(), tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(NewsDataset(val_df.statement.to_numpy(), val_df.label.to_numpy(), tokenizer, MAX_LEN), batch_size=BATCH_SIZE)
    
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
    optimizer = AdamW(model.parameters(), lr=LR)
    
    best_f1 = 0
    for epoch in range(EPOCHS):
        model.train()
        for d in train_loader:
            input_ids, mask, labels = d["input_ids"].to(device), d["attention_mask"].to(device), d["labels"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=mask, labels=labels)
            outputs.loss.backward(); optimizer.step(); optimizer.zero_grad()
        
        f1 = eval_model(model, val_loader, device)
        print(f"Epoch {epoch+1} F1: {f1:.4f}")
        if f1 > best_f1:
            best_f1 = f1; model.save_pretrained('models/best_model'); tokenizer.save_pretrained('models/best_model')
    
    print("Training complete.")

if __name__ == "__main__":
    main()

## 3. Run Pipeline
First, we preprocess the data, then we start the training.

In [ ]:
!python src/preprocess.py
!python src/train.py

## 4. Download Trained Model
After training, zip the `models/best_model` folder and download it to use in your local setup.

In [ ]:
!zip -r best_model.zip models/best_model
from google.colab import files
files.download('best_model.zip')